# LT3 — Random Forest cho dữ liệu FANET tách theo kịch bản

LT3 kế thừa luồng nghiên cứu của LT2: đọc dữ liệu → tạo đặc trưng cửa sổ → audit → chia tập → so sánh baseline → đánh giá Random Forest → phân tích đặc trưng → lưu bundle.

Các thay đổi quan trọng:

- Đọc nhiều CSV độc lập, mỗi CSV tương ứng đúng một lần mô phỏng.
- Kiểm tra manifest và provenance trước khi gộp dữ liệu trong bộ nhớ.
- Chia train/validation/test theo `replicate_group_id`, không chia các cửa sổ của cùng một flow/run sang nhiều tập.
- Hỗ trợ `multiclass`, `binary_udp`, `binary_tcp`.
- Random Forest là mô hình chính; các mô hình còn lại chỉ là baseline.
- Không ghi đè kết quả cũ: mỗi lần chạy tạo thư mục có timestamp.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 100)

## 1. Cấu hình duy nhất cần chỉnh

`DATA_ROOT` phải chứa ba thư mục `normal`, `udp_flooding`, `tcp_syn_flooding`. Mỗi CSV là một run và có một manifest JSON cùng tên.

In [ ]:
RANDOM_STATE = 42

PROJECT_DIR = Path(r"F:\NCKH_ICTU_2025\Using-machine-learning-to-detect-attacks-in-a-network-of-unmanned-aerial-vehicles-FANET")
DATA_ROOT = PROJECT_DIR / "data" / "raw"
OUTPUT_ROOT = PROJECT_DIR / "fanet_lt3_outputs"

# multiclass | binary_udp | binary_tcp
TRAINING_MODE = "multiclass"

FILE_GLOB = "**/*.csv"
REQUIRE_MANIFEST = True
REQUIRE_PAIRED_REPLICATES = True
COMPUTE_SHA256 = True
RUN_EXPENSIVE_STUDIES = False

TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
TRAIN_MAX_ROWS_PER_CLASS = 200_000

ALLOWED_LABELS = ("normal", "udp_flooding", "tcp_syn_flooding")
LABEL_ALIASES = {
    "normal": "normal",
    "dos_udp_flood": "udp_flooding",
    "udp_flood": "udp_flooding",
    "udp_flooding": "udp_flooding",
    "tcp_syn_flood": "tcp_syn_flooding",
    "syn_flood": "tcp_syn_flooding",
    "tcp_syn_flooding": "tcp_syn_flooding",
}

MODE_LABELS = {
    "multiclass": ("normal", "udp_flooding", "tcp_syn_flooding"),
    "binary_udp": ("normal", "udp_flooding"),
    "binary_tcp": ("normal", "tcp_syn_flooding"),
}

MODE_ATTACK_TYPES = {
    "multiclass": ("normal", "udp_flooding", "tcp_syn_flooding"),
    "binary_udp": ("normal", "udp_flooding"),
    "binary_tcp": ("normal", "tcp_syn_flooding"),
}

REQUIRED_COLUMNS = {
    "scenario_id", "replicate_group_id", "run_id", "seed",
    "mobility_seed", "traffic_seed", "executable", "attack_type",
    "attack_phase", "attack_label", "window_id", "time_s", "flow_id",
    "srcNode", "dstNode", "srcIp", "dstIp", "srcPort", "dstPort",
    "protocol", "txPackets", "rxPackets", "lostPackets", "txBytes",
    "rxBytes", "delay_ms", "jitter_ms", "syn_count", "ack_count",
    "rst_count", "speed", "neighbors",
}

OBSERVATION_KEY = [
    "scenario_id", "run_id", "window_id", "time_s", "flow_id"
]

FEATURES = [
    "tx_pps", "rx_pps", "drop_pps", "tx_bps", "rx_bps",
    "window_pdr", "window_plr", "tx_mean_pkt_bytes",
    "rx_mean_pkt_bytes", "delay_ms", "jitter_ms", "speed",
    "neighbors", "syn_pps", "ack_pps", "rst_pps",
    "syn_ack_ratio_window",
]

if TRAINING_MODE not in MODE_LABELS:
    raise ValueError(f"TRAINING_MODE không hợp lệ: {TRAINING_MODE}")

## 2. Hàm hỗ trợ provenance và đầu ra

In [ ]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(
        json.dumps(json_safe(payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_label_series(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower().map(LABEL_ALIASES)

## 3. Khám phá và kiểm định từng run

Không dùng `pd.concat` mù. Mỗi file phải vượt qua kiểm định riêng trước khi được đưa vào dataset chung.

In [ ]:
def discover_run_files(data_root: Path, pattern: str = FILE_GLOB) -> list[Path]:
    if not data_root.is_dir():
        raise FileNotFoundError(
            f"Không tìm thấy DATA_ROOT: {data_root}\n"
            "Hãy sinh dữ liệu NS-3 vào data/raw/<attack_type>/*.csv trước."
        )
    files = sorted(path for path in data_root.glob(pattern) if path.is_file())
    if not files:
        raise FileNotFoundError(f"Không tìm thấy CSV trong {data_root}")
    return files


def load_sidecar_manifest(csv_path: Path) -> dict[str, Any] | None:
    manifest_path = csv_path.with_suffix(".json")
    if not manifest_path.is_file():
        return None
    return json.loads(manifest_path.read_text(encoding="utf-8"))


def audit_and_load_run(csv_path: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    errors: list[str] = []
    warnings: list[str] = []
    df = pd.read_csv(csv_path, low_memory=False)

    missing = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing:
        errors.append(f"Thiếu cột: {missing}")
        return df, {
            "file": str(csv_path), "passed": False, "rows": len(df),
            "errors": errors, "warnings": warnings,
        }

    df["attack_label"] = normalize_label_series(df["attack_label"])
    df["attack_type"] = normalize_label_series(df["attack_type"])

    if df["attack_label"].isna().any():
        errors.append("Có attack_label không thuộc ba nhãn chuẩn")
    if df["attack_type"].isna().any():
        errors.append("Có attack_type không thuộc ba loại kịch bản chuẩn")

    exact_duplicates = int(df.duplicated().sum())
    duplicate_keys = int(df.duplicated(OBSERVATION_KEY, keep=False).sum())
    if exact_duplicates:
        errors.append(f"Có {exact_duplicates} dòng trùng hoàn toàn")
    if duplicate_keys:
        errors.append(f"Có {duplicate_keys} dòng trùng khóa {OBSERVATION_KEY}")

    singleton_columns = [
        "scenario_id", "replicate_group_id", "run_id", "seed",
        "mobility_seed", "traffic_seed", "executable", "attack_type",
    ]
    for column in singleton_columns:
        if df[column].nunique(dropna=False) != 1:
            errors.append(f"Một file phải có đúng một giá trị {column}")

    if not errors:
        attack_type = str(df["attack_type"].iloc[0])
        labels_in_file = set(df["attack_label"].dropna().astype(str).unique())
        allowed_for_scenario = {"normal"} if attack_type == "normal" else {"normal", attack_type}
        forbidden = labels_in_file - allowed_for_scenario
        if forbidden:
            errors.append(
                f"Kịch bản {attack_type} chứa nhãn không hợp lệ: {sorted(forbidden)}"
            )
        if attack_type != "normal" and attack_type not in labels_in_file:
            errors.append(f"Kịch bản {attack_type} không có dòng tấn công tương ứng")

    manifest = load_sidecar_manifest(csv_path)
    if REQUIRE_MANIFEST and manifest is None:
        errors.append("Thiếu manifest JSON cùng tên với CSV")
    if manifest is not None and not errors:
        for field in ("scenario_id", "replicate_group_id", "run_id", "attack_type"):
            if field not in manifest:
                errors.append(f"Manifest thiếu trường {field}")
                continue
            csv_value = str(df[field].iloc[0])
            manifest_value = str(manifest[field])
            if csv_value != manifest_value:
                errors.append(
                    f"Manifest/CSV không khớp {field}: {manifest_value!r} != {csv_value!r}"
                )

    df["source_file"] = str(csv_path)
    report = {
        "file": str(csv_path),
        "passed": not errors,
        "rows": int(len(df)),
        "exact_duplicate_rows": exact_duplicates,
        "duplicate_key_rows": duplicate_keys,
        "sha256": sha256_file(csv_path) if COMPUTE_SHA256 else None,
        "errors": errors,
        "warnings": warnings,
    }
    if not errors:
        report.update({
            "scenario_id": str(df["scenario_id"].iloc[0]),
            "replicate_group_id": str(df["replicate_group_id"].iloc[0]),
            "run_id": str(df["run_id"].iloc[0]),
            "attack_type": str(df["attack_type"].iloc[0]),
            "label_counts": df["attack_label"].value_counts().to_dict(),
        })
    return df, report


def load_validated_runs(files: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    frames: list[pd.DataFrame] = []
    reports: list[dict[str, Any]] = []
    for csv_path in files:
        frame, report = audit_and_load_run(csv_path)
        frames.append(frame)
        reports.append(report)

    errors = [
        f"{report['file']}: {message}"
        for report in reports
        for message in report["errors"]
    ]
    if errors:
        return pd.DataFrame(), pd.DataFrame(reports), {
            "passed": False, "errors": errors, "warnings": []
        }

    combined = pd.concat(frames, ignore_index=True, copy=False)
    run_index = pd.DataFrame(reports)

    duplicate_run_files = (
        combined.groupby(["scenario_id", "run_id"])["source_file"].nunique() > 1
    )
    if duplicate_run_files.any():
        errors.append("Cùng scenario_id/run_id xuất hiện trong nhiều file")

    combined_duplicate_keys = int(
        combined.duplicated(OBSERVATION_KEY, keep=False).sum()
    )
    if combined_duplicate_keys:
        errors.append(
            f"Dataset gộp có {combined_duplicate_keys} dòng trùng khóa quan sát"
        )

    coverage = (
        combined.groupby("replicate_group_id")["attack_type"]
        .agg(lambda values: sorted(set(values)))
    )
    expected = set(ALLOWED_LABELS)
    incomplete_groups = {
        str(group): values
        for group, values in coverage.items()
        if set(values) != expected
    }
    if REQUIRE_PAIRED_REPLICATES and incomplete_groups:
        errors.append(
            f"Có {len(incomplete_groups)} replicate_group_id không đủ normal/UDP/TCP"
        )

    audit = {
        "passed": not errors,
        "file_count": len(files),
        "row_count": int(len(combined)),
        "run_count": int(combined.groupby(["scenario_id", "run_id"]).ngroups),
        "replicate_group_count": int(combined["replicate_group_id"].nunique()),
        "combined_duplicate_key_rows": combined_duplicate_keys,
        "label_counts": combined["attack_label"].value_counts().to_dict(),
        "attack_type_counts": combined["attack_type"].value_counts().to_dict(),
        "incomplete_replicate_groups": incomplete_groups,
        "errors": errors,
        "warnings": [],
    }
    return combined, run_index, audit

In [ ]:
run_files = discover_run_files(DATA_ROOT)
raw_df, run_index, raw_audit = load_validated_runs(run_files)

print(f"Số file phát hiện: {len(run_files)}")
display(run_index[[column for column in [
    "passed", "attack_type", "scenario_id", "replicate_group_id",
    "run_id", "rows", "file"
] if column in run_index.columns]])

print("\nKết quả audit tổng:")
display(pd.DataFrame({
    "field": ["passed", "file_count", "row_count", "run_count", "replicate_group_count"],
    "value": [
        raw_audit.get("passed"), raw_audit.get("file_count"),
        raw_audit.get("row_count"), raw_audit.get("run_count"),
        raw_audit.get("replicate_group_count"),
    ],
}))

if raw_audit["errors"]:
    print("ERRORS:")
    for message in raw_audit["errors"]:
        print(" -", message)
    raise RuntimeError("STOP: dữ liệu nguồn không vượt qua kiểm định")

## 4. Tạo đặc trưng theo cửa sổ

Các counter tích lũy được lấy hiệu trong phạm vi `(scenario_id, run_id, flow_id)`. Việc thêm khóa run vào phép nhóm ngăn dữ liệu của hai mô phỏng khác nhau bị nối thành cùng một chuỗi thời gian.

In [ ]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator.div(denominator.replace(0, np.nan))


def build_window_features(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any]]:
    data = df.sort_values(
        ["scenario_id", "run_id", "flow_id", "time_s"],
        kind="mergesort",
    ).copy()
    group_columns = ["scenario_id", "run_id", "flow_id"]
    grouped = data.groupby(group_columns, sort=False, dropna=False)

    data["dt_s"] = grouped["time_s"].diff()
    counter_map = {
        "txPackets": "d_tx_packets",
        "rxPackets": "d_rx_packets",
        "lostPackets": "d_lost_packets",
        "txBytes": "d_tx_bytes",
        "rxBytes": "d_rx_bytes",
        "syn_count": "d_syn",
        "ack_count": "d_ack",
        "rst_count": "d_rst",
    }

    reset_counts = {}
    for source, target in counter_map.items():
        delta = grouped[source].diff()
        reset_counts[source] = int(delta.lt(0).sum())
        data[target] = delta.mask(delta < 0)

    before = len(data)
    data = data.loc[data["dt_s"].gt(0)].copy()

    data["tx_pps"] = safe_divide(data["d_tx_packets"], data["dt_s"])
    data["rx_pps"] = safe_divide(data["d_rx_packets"], data["dt_s"])
    data["drop_pps"] = safe_divide(data["d_lost_packets"], data["dt_s"])
    data["tx_bps"] = 8.0 * safe_divide(data["d_tx_bytes"], data["dt_s"])
    data["rx_bps"] = 8.0 * safe_divide(data["d_rx_bytes"], data["dt_s"])
    data["window_pdr"] = safe_divide(data["d_rx_packets"], data["d_tx_packets"])
    data["window_plr"] = safe_divide(data["d_lost_packets"], data["d_tx_packets"])
    data["tx_mean_pkt_bytes"] = safe_divide(data["d_tx_bytes"], data["d_tx_packets"])
    data["rx_mean_pkt_bytes"] = safe_divide(data["d_rx_bytes"], data["d_rx_packets"])
    data["syn_pps"] = safe_divide(data["d_syn"], data["dt_s"])
    data["ack_pps"] = safe_divide(data["d_ack"], data["dt_s"])
    data["rst_pps"] = safe_divide(data["d_rst"], data["dt_s"])
    data["syn_ack_ratio_window"] = safe_divide(data["d_syn"], data["d_ack"])

    data[FEATURES] = data[FEATURES].replace([np.inf, -np.inf], np.nan)
    feature_audit = {
        "input_rows": int(before),
        "output_rows": int(len(data)),
        "dropped_first_or_nonpositive_dt_rows": int(before - len(data)),
        "counter_reset_counts": reset_counts,
        "nan_counts": data[FEATURES].isna().sum().to_dict(),
        "negative_rate_rows": int(
            data[["tx_pps", "rx_pps", "drop_pps", "tx_bps", "rx_bps"]]
            .lt(0).any(axis=1).sum()
        ),
    }
    return data, feature_audit


window_df, feature_audit = build_window_features(raw_df)

print("Kích thước sau tạo đặc trưng:", window_df.shape)
display(pd.DataFrame({
    "feature": FEATURES,
    "missing": [feature_audit["nan_counts"][feature] for feature in FEATURES],
}))

if feature_audit["negative_rate_rows"]:
    raise RuntimeError("STOP: phát hiện đặc trưng tốc độ âm")
if any(feature_audit["counter_reset_counts"].values()):
    raise RuntimeError(
        "STOP: counter bị reset trong cùng một run/flow; cần kiểm tra bộ sinh CSV"
    )

## 5. Chọn bài toán học

Với bài toán nhị phân, LT3 chỉ lấy các kịch bản tương ứng. Ví dụ `binary_udp` không lấy những dòng `normal` nằm trong kịch bản TCP, tránh phụ thuộc chéo giữa hai cuộc tấn công.

In [ ]:
selected_attack_types = set(MODE_ATTACK_TYPES[TRAINING_MODE])
selected_labels = list(MODE_LABELS[TRAINING_MODE])

model_df = window_df.loc[
    window_df["attack_type"].isin(selected_attack_types)
    & window_df["attack_label"].isin(selected_labels)
].reset_index(drop=True)

present_labels = set(model_df["attack_label"].unique())
missing_labels = set(selected_labels) - present_labels
if missing_labels:
    raise RuntimeError(f"Thiếu lớp cho {TRAINING_MODE}: {sorted(missing_labels)}")

print("Training mode:", TRAINING_MODE)
print("Nhãn:", selected_labels)
print("Số replicate group:", model_df["replicate_group_id"].nunique())
display(pd.crosstab(model_df["attack_type"], model_df["attack_label"]))

## 6. Chia train/validation/test theo replicate

Một `replicate_group_id` chỉ được xuất hiện trong đúng một tập. Đây là cổng chống leakage chính của LT3.

In [ ]:
def split_indices_once(
    indices: np.ndarray,
    y: pd.Series,
    groups: pd.Series,
    test_size: float,
    random_state: int,
) -> tuple[np.ndarray, np.ndarray]:
    splitter = GroupShuffleSplit(
        n_splits=1, test_size=test_size, random_state=random_state
    )
    left_pos, right_pos = next(
        splitter.split(indices, y.iloc[indices], groups.iloc[indices])
    )
    return indices[left_pos], indices[right_pos]


def group_train_validation_test_split(
    df: pd.DataFrame,
    labels: list[str],
    max_attempts: int = 200,
) -> dict[str, np.ndarray]:
    y = df["attack_label"].reset_index(drop=True)
    groups = df["replicate_group_id"].astype("string").reset_index(drop=True)
    indices = np.arange(len(df))
    required = set(labels)

    for attempt in range(max_attempts):
        seed = RANDOM_STATE + attempt
        train_val, test = split_indices_once(
            indices, y, groups, TEST_SIZE, seed
        )
        relative_val_size = VALIDATION_SIZE / (1.0 - TEST_SIZE)
        train, validation = split_indices_once(
            train_val, y, groups, relative_val_size, seed + 10_000
        )
        splits = {"train": train, "validation": validation, "test": test}

        if not all(set(y.iloc[idx].unique()) == required for idx in splits.values()):
            continue

        group_sets = {
            name: set(groups.iloc[idx].astype(str))
            for name, idx in splits.items()
        }
        overlap = (
            (group_sets["train"] & group_sets["validation"])
            | (group_sets["train"] & group_sets["test"])
            | (group_sets["validation"] & group_sets["test"])
        )
        if not overlap:
            return splits

    raise RuntimeError(
        "Không thể tạo ba tập độc lập có đủ lớp. Cần sinh thêm replicate_group_id."
    )


split_indices = group_train_validation_test_split(model_df, selected_labels)
model_df["split"] = ""
for split_name, indices in split_indices.items():
    model_df.loc[indices, "split"] = split_name

split_group_table = (
    model_df[["replicate_group_id", "split"]]
    .drop_duplicates()
    .sort_values(["split", "replicate_group_id"])
)

assert split_group_table.groupby("replicate_group_id")["split"].nunique().max() == 1

display(pd.crosstab(model_df["split"], model_df["attack_label"]))
display(
    split_group_table.groupby("split")["replicate_group_id"]
    .nunique().rename("replicate_groups").to_frame()
)

## 7. Chuẩn bị tập học

Việc giới hạn số dòng chỉ áp dụng cho train. Validation và test không bị lấy mẫu. Imputer và scaler nằm trong Pipeline nên chỉ được fit từ train.

In [ ]:
train_df = model_df.loc[model_df["split"] == "train"].copy()
validation_df = model_df.loc[model_df["split"] == "validation"].copy()
test_df = model_df.loc[model_df["split"] == "test"].copy()


def cap_rows_per_class(
    df: pd.DataFrame,
    max_rows: int | None,
    random_state: int,
) -> pd.DataFrame:
    if max_rows is None:
        return df
    parts = []
    for _, class_df in df.groupby("attack_label", sort=False):
        if len(class_df) > max_rows:
            class_df = class_df.sample(n=max_rows, random_state=random_state)
        parts.append(class_df)
    return pd.concat(parts, ignore_index=True).sample(
        frac=1.0, random_state=random_state
    ).reset_index(drop=True)


train_fit_df = cap_rows_per_class(
    train_df, TRAIN_MAX_ROWS_PER_CLASS, RANDOM_STATE
)

print("Train trước giới hạn:", len(train_df))
print("Train dùng để fit:", len(train_fit_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

## 8. Random Forest chính và các baseline

Không tự động đổi đề tài sang mô hình khác khi baseline có điểm cao hơn. Random Forest được cố định là mô hình chính; bảng baseline dùng để so sánh khoa học.

In [ ]:
def make_candidates(random_state: int = RANDOM_STATE) -> dict[str, Pipeline]:
    common_imputer = SimpleImputer(strategy="median", add_indicator=True)

    return {
        "Dummy": Pipeline([
            ("imputer", clone(common_imputer)),
            ("classifier", DummyClassifier(strategy="prior")),
        ]),
        "LogisticRegression": Pipeline([
            ("imputer", clone(common_imputer)),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=2_000,
                class_weight="balanced",
                random_state=random_state,
            )),
        ]),
        "DecisionTree": Pipeline([
            ("imputer", clone(common_imputer)),
            ("classifier", DecisionTreeClassifier(
                max_depth=12,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=random_state,
            )),
        ]),
        "RandomForest": Pipeline([
            ("imputer", clone(common_imputer)),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced_subsample",
                bootstrap=True,
                oob_score=True,
                random_state=random_state,
                n_jobs=-1,
            )),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", clone(common_imputer)),
            ("classifier", ExtraTreesClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced",
                random_state=random_state,
                n_jobs=-1,
            )),
        ]),
    }


def score_predictions(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(
            precision_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
    }


def evaluate_candidates(
    train_data: pd.DataFrame,
    validation_data: pd.DataFrame,
    candidates: dict[str, Pipeline],
) -> tuple[pd.DataFrame, dict[str, Pipeline]]:
    rows = []
    fitted = {}
    for name, candidate in candidates.items():
        model = clone(candidate).fit(
            train_data[FEATURES], train_data["attack_label"]
        )
        prediction = model.predict(validation_data[FEATURES])
        rows.append({"model": name, **score_predictions(
            validation_data["attack_label"], prediction
        )})
        fitted[name] = model
    return (
        pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True),
        fitted,
    )

In [ ]:
candidates = make_candidates()
validation_results, fitted_validation_models = evaluate_candidates(
    train_fit_df, validation_df, candidates
)

display(validation_results)

PRIMARY_MODEL_NAME = "RandomForest"
primary_validation_model = fitted_validation_models[PRIMARY_MODEL_NAME]
print("Mô hình chính cố định:", PRIMARY_MODEL_NAME)
print(
    "Validation Macro-F1:",
    validation_results.loc[
        validation_results["model"] == PRIMARY_MODEL_NAME, "macro_f1"
    ].iloc[0],
)

## 9. Đánh giá test đúng một lần

Sau khi kiến trúc và tham số Random Forest đã cố định bằng validation, mô hình được fit lại trên train + validation rồi mới mở test.

In [ ]:
train_validation_df = pd.concat(
    [train_df, validation_df], ignore_index=True, copy=False
)
train_validation_fit_df = cap_rows_per_class(
    train_validation_df,
    TRAIN_MAX_ROWS_PER_CLASS,
    RANDOM_STATE,
)

evaluation_model = clone(candidates[PRIMARY_MODEL_NAME]).fit(
    train_validation_fit_df[FEATURES],
    train_validation_fit_df["attack_label"],
)
test_prediction = evaluation_model.predict(test_df[FEATURES])
test_metrics = score_predictions(test_df["attack_label"], test_prediction)

print("Test metrics:")
display(pd.DataFrame([test_metrics]))

classification = classification_report(
    test_df["attack_label"],
    test_prediction,
    labels=selected_labels,
    output_dict=True,
    zero_division=0,
)
display(pd.DataFrame(classification).T)

matrix = confusion_matrix(
    test_df["attack_label"], test_prediction, labels=selected_labels
)
matrix_df = pd.DataFrame(
    matrix,
    index=[f"true:{label}" for label in selected_labels],
    columns=[f"pred:{label}" for label in selected_labels],
)
display(matrix_df)

## 10. Độ ổn định giữa các replicate test

Điểm trung bình theo dòng có thể bị chi phối bởi những run dài. Bảng dưới tính riêng từng `replicate_group_id` rồi báo trung bình và độ lệch chuẩn.

In [ ]:
test_scored = test_df[["replicate_group_id", "attack_label"]].copy()
test_scored["prediction"] = test_prediction

per_replicate_rows = []
for replicate_group_id, group in test_scored.groupby("replicate_group_id"):
    per_replicate_rows.append({
        "replicate_group_id": replicate_group_id,
        **score_predictions(group["attack_label"], group["prediction"].to_numpy()),
    })

per_replicate_metrics = pd.DataFrame(per_replicate_rows)
display(per_replicate_metrics)
display(per_replicate_metrics.drop(columns="replicate_group_id").agg(["mean", "std"]))

## 11. Độ quan trọng đặc trưng

Permutation importance được tính trên test chưa từng tham gia huấn luyện. Nếu test quá lớn, chỉ lấy mẫu để kiểm soát thời gian tính.

In [ ]:
importance_sample = test_df
if len(importance_sample) > 100_000:
    importance_sample = importance_sample.sample(
        n=100_000, random_state=RANDOM_STATE
    )

permutation = permutation_importance(
    evaluation_model,
    importance_sample[FEATURES],
    importance_sample["attack_label"],
    scoring="f1_macro",
    n_repeats=5,
    random_state=RANDOM_STATE,
    # Một tiến trình ổn định hơn với Jupyter/Windows và các virtual environment.
    n_jobs=1,
)

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": permutation.importances_mean,
    "importance_std": permutation.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance_df)

## 12. Ablation và số cây — tùy chọn

Các nghiên cứu tốn thời gian mặc định bị tắt. Chỉ bật sau khi pipeline chính đã PASS.

In [ ]:
def run_optional_studies(
    train_data: pd.DataFrame,
    validation_data: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_sets = {
        "traffic_only": [
            "tx_pps", "rx_pps", "drop_pps", "tx_bps", "rx_bps",
            "window_pdr", "window_plr", "tx_mean_pkt_bytes",
            "rx_mean_pkt_bytes", "delay_ms", "jitter_ms",
        ],
        "traffic_plus_mobility": [
            "tx_pps", "rx_pps", "drop_pps", "tx_bps", "rx_bps",
            "window_pdr", "window_plr", "tx_mean_pkt_bytes",
            "rx_mean_pkt_bytes", "delay_ms", "jitter_ms", "speed", "neighbors",
        ],
        "all_features": FEATURES,
    }

    ablation_rows = []
    for name, feature_set in feature_sets.items():
        model = clone(candidates[PRIMARY_MODEL_NAME]).fit(
            train_data[feature_set], train_data["attack_label"]
        )
        pred = model.predict(validation_data[feature_set])
        ablation_rows.append({
            "feature_set": name,
            "feature_count": len(feature_set),
            **score_predictions(validation_data["attack_label"], pred),
        })

    tree_rows = []
    for n_trees in (100, 300, 500):
        candidate = clone(candidates[PRIMARY_MODEL_NAME])
        candidate.set_params(classifier__n_estimators=n_trees)
        candidate.fit(train_data[FEATURES], train_data["attack_label"])
        pred = candidate.predict(validation_data[FEATURES])
        tree_rows.append({
            "n_estimators": n_trees,
            **score_predictions(validation_data["attack_label"], pred),
        })
    return pd.DataFrame(ablation_rows), pd.DataFrame(tree_rows)


if RUN_EXPENSIVE_STUDIES:
    ablation_results, tree_count_results = run_optional_studies(
        train_fit_df, validation_df
    )
    display(ablation_results)
    display(tree_count_results)
else:
    ablation_results = pd.DataFrame()
    tree_count_results = pd.DataFrame()
    print("Đã bỏ qua ablation/tree-count. Đổi RUN_EXPENSIVE_STUDIES=True để chạy.")

## 13. Lưu kết quả không ghi đè

Bundle chứa mô hình, danh sách đặc trưng, nhãn và chế độ huấn luyện. Manifest ghi lại hash của từng CSV đầu vào để có thể tái lập thí nghiệm.

In [ ]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = OUTPUT_ROOT / f"{TRAINING_MODE}_{run_timestamp}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

bundle = {
    "model": evaluation_model,
    "features": FEATURES,
    "labels": selected_labels,
    "training_mode": TRAINING_MODE,
    "group_column": "replicate_group_id",
    "feature_engineering_version": "window_delta_lt3_v1",
}
joblib.dump(bundle, OUTPUT_DIR / "random_forest_bundle.joblib")

validation_results.to_csv(
    OUTPUT_DIR / "validation_model_comparison.csv", index=False
)
matrix_df.to_csv(OUTPUT_DIR / "test_confusion_matrix.csv")
importance_df.to_csv(OUTPUT_DIR / "permutation_importance.csv", index=False)
per_replicate_metrics.to_csv(
    OUTPUT_DIR / "test_metrics_per_replicate.csv", index=False
)
split_group_table.to_csv(OUTPUT_DIR / "split_groups.csv", index=False)
run_index.to_csv(OUTPUT_DIR / "input_run_index.csv", index=False)

if not ablation_results.empty:
    ablation_results.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)
if not tree_count_results.empty:
    tree_count_results.to_csv(OUTPUT_DIR / "tree_count_results.csv", index=False)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_status": "publishable_candidate",
    "training_mode": TRAINING_MODE,
    "primary_model": PRIMARY_MODEL_NAME,
    "features": FEATURES,
    "labels": selected_labels,
    "group_column": "replicate_group_id",
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "validation_size": VALIDATION_SIZE,
    "train_max_rows_per_class": TRAIN_MAX_ROWS_PER_CLASS,
    "raw_audit": raw_audit,
    "feature_audit": feature_audit,
    "test_metrics": test_metrics,
    "classification_report": classification,
    "oob_score": float(
        evaluation_model.named_steps["classifier"].oob_score_
    ),
    "input_files": run_index.to_dict(orient="records"),
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}
write_json(OUTPUT_DIR / "experiment_manifest.json", manifest)

print("Hoàn tất. Output:", OUTPUT_DIR)

## 14. Cách sử dụng LT3

1. Sinh nhiều run độc lập cho normal, UDP flooding và TCP SYN flooding.
2. Đặt CSV/manifest vào `data/raw/<attack_type>/`.
3. Chạy `TRAINING_MODE="multiclass"` để lấy kết quả chính.
4. Chạy lại với `binary_udp` và `binary_tcp` để phân tích từng cuộc tấn công.
5. Không so sánh kết quả giữa các mode nếu chúng không dùng cùng tập `replicate_group_id`.

LT3 không sửa nhãn sai và không tự tạo provenance cho dữ liệu cũ. Nếu audit FAIL, phải sửa ở bộ sinh dữ liệu NS-3.